In [0]:
%sql
USE CATALOG industry

# Clean the Raw Data

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW staged_location AS
WITH clean_data AS (
  SELECT
    country_id,
    CAST(SPLIT(country_id, "-")[1] AS BIGINT) AS id,
    CASE
      WHEN country_name = 'DE' THEN 'Germany'
      WHEN country_name in ('US', 'USA') THEN 'United States'
      ELSE TRIM(country_name)
    END AS country_name,
    etl_filename
  FROM
    bronze.location
),
filter_data AS (
  SELECT
    id as location_id,
    country_name as country_etl_business_key,
    country_id AS etl_business_key,
    etl_filename
  FROM
    clean_data
  WHERE
    country_name != ""
    AND country_name IS NOT NULL
)
SELECT
  location_id,
  COALESCE(country.country_sk, -1) AS country_sk,
  filter_data.etl_business_key,
  SHA2(CONCAT(location_id, COALESCE(country.country_sk, -1)), 256) AS etl_record_hash,
  filter_data.etl_filename
FROM
  filter_data
    LEFT JOIN silver.country
      ON country_etl_business_key = country.etl_business_key

# MERGE TO Silver Table

In [0]:
%sql
MERGE INTO
  silver.location as tgt
USING
  staged_location as src
ON
  tgt.etl_business_key = src.etl_business_key
  AND tgt.etl_record_hash <> src.etl_record_hash
WHEN MATCHED THEN UPDATE SET
  tgt.country_sk = src.country_sk,
  tgt.etl_record_hash = src.etl_record_hash,
  tgt.etl_update_timestamp = NOW()
WHEN NOT MATCHED THEN INSERT (
    tgt.location_id,
    tgt.country_sk,
    tgt.etl_business_key,
    tgt.etl_record_hash,
    tgt.etl_filename
  )
  VALUES (
    src.location_id,
    src.country_sk,
    src.etl_business_key,
    src.etl_record_hash,
    src.etl_filename
  )